In [1]:
# ruff: noqa: F401, F403

import dataclasses
import os
import subprocess
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [2]:
file_paths = [
    # "/Users/denys/Documents/gokarting-ui/GH010219.MP4",
    # "/Users/denys/Documents/gokarting-ui/GH020219.MP4",
    # "/Users/denys/Documents/gokarting-ui/GH030219.MP4",
    # "/Users/denys/Documents/gokarting-ui/GH040219.MP4",
    # "/Users/denys/Documents/gokarting-ui/GH050219.MP4",
    # "/Users/denys/Documents/video-dump/GX010284.MP4",
    "/Users/denys/Documents/video-dump/GX010295.MP4",
    "/Users/denys/Documents/video-dump/GX020295.MP4",
    # ]
    # file_paths = [
    # "/Users/denys/Downloads/GX010108.MP4",
    # "/Users/denys/Downloads/GX020108.MP4",
    # "/Users/denys/Downloads/GX030108.MP4",
    # "/Users/denys/Downloads/GX040108.MP4",
]

In [3]:
@dataclasses.dataclass
class Source:
    start_timestamp: pd.Timestamp
    end_timestamp: pd.Timestamp
    filepath: str


@dataclasses.dataclass
class Session:
    cs: CoordinateSystem
    laps: Laps
    sources: list[Source]

    def locate_timestamp(self, timestamp: pd.Timestamp) -> tuple[str, float]:
        """
        Locates timestamp from all files in the session.
        """

        for source in self.sources:
            if source.start_timestamp <= timestamp <= source.end_timestamp:
                offset = (timestamp - source.start_timestamp).total_seconds()
                return source.filepath, offset
        raise ValueError(f"Timestamp {timestamp} not found in any source.")

    

In [4]:
single_files = [GPMFSource(f) for f in file_paths]

samples = []
start_of_file = {}
end_of_file = {}


for fn, f in zip(file_paths, single_files):
    total_duration = f.get_total_duration()

    def on_sample(s, _, _2):
        if fn not in start_of_file:
            start_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
        end_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
        if s.full_speed > 3:
            samples.append(s)

    while not f.is_end():
        f.read_samples(on_sample)
        f.next()


mean_point = GPSSample(
    lat=np.mean([s.lat for s in samples]),
    lon=np.mean([s.lon for s in samples]),
    altitude=np.mean([s.altitude for s in samples]),
)


cs = CoordinateSystem(mean_point)

In [5]:
import pacer._pacer


pacer._pacer.__file__

'/Users/denys/dev/pacer/.pixi/envs/default/lib/python3.13/site-packages/pacer/_pacer.cpython-313-darwin.so'

In [6]:
samples[0]

GPSSample(lat=51.375612, lon=-0.361308, altitude=11.506000, full_speed=3.197000, ground_speed=3.000000, timestamp_ms=1765637078695)

In [7]:
start_time = pd.to_datetime(samples[0].timestamp_ms, unit="ms")

def locate_timestamp(timestamp: pd.Timestamp | float):
    if isinstance(timestamp, float):
        timestamp = start_time + pd.to_timedelta(timestamp, unit="s")
    for f in file_paths:
        if start_of_file[f] <= timestamp < end_of_file[f]:
            return f, timestamp - start_of_file[f]

In [8]:
attributes: list[str] = [
    attr for attr in samples[0].__dir__() if not attr.startswith("_")
]
df = pd.DataFrame(
    [
        {k: getattr(s, k) for k in attributes}
        | {p: getattr(cs.local(s), p) for p in "xyz"}
        for s in samples
    ]
)
df = df.assign(timestamp=lambda d: pd.to_datetime(d["timestamp_ms"], unit="ms"))
df = df.assign(
    dist_prev=lambda d: (
        d["x"].diff().pow(2) + d["y"].diff().pow(2) + d["z"].diff().pow(2)
    ).pow(0.5),
    time_prev=lambda d: d["dist_prev"] / d["full_speed"],
)

dataset = df.iloc[200:].loc[
    lambda d: (d["timestamp"] == d["timestamp"].shift(+1))
    | (d["timestamp"] == d["timestamp"].shift(-1))
]
dataset.pipe(lambda d: px.scatter(d, y="time_prev"))

In [9]:
px.scatter(df["time_prev"])

In [10]:
dataset = dataset.assign(
    di=lambda d: (d["time_prev"] / d["time_prev"].median()).round()
)
total_points = dataset["di"].sum()
total_time = dataset["timestamp"].max() - dataset["timestamp"].min()
single_step = total_time / (total_points - 9)
dataset = dataset.assign(
    timestamp_corrected=lambda d: dataset["timestamp"].iloc[0]
    + (d["di"] * single_step).cumsum()
)
dataset

df = df.assign(
    timestamp_corrected=dataset["timestamp_corrected"]
    .reindex(df.index)
    .fillna(df["timestamp"])
)
df["timestamp_corrected"]

0       2025-12-13 14:44:38.695000000
1       2025-12-13 14:44:38.695000000
2       2025-12-13 14:44:39.685000000
3       2025-12-13 14:44:39.685000000
4       2025-12-13 14:44:39.685000000
                     ...             
36351   2025-12-13 15:19:49.923984300
36352   2025-12-13 15:19:49.978984272
36353   2025-12-13 15:19:50.033984244
36354   2025-12-13 15:19:50.088984216
36355   2025-12-13 15:19:50.143984188
Name: timestamp_corrected, Length: 36356, dtype: datetime64[ns]

In [11]:
px.scatter((df["timestamp"] - df["timestamp_corrected"]).dt.total_seconds())

In [12]:
laps = Laps()

for _, row in df.iterrows():
    s = GPSSample(
        lat=row["lat"],
        lon=row["lon"],
        altitude=row["altitude"],
        full_speed=row["full_speed"],
        timestamp_ms=row["timestamp_corrected"].value // 10**6,
    )
    t = (row["timestamp_corrected"] - start_time).total_seconds()
    laps.add_point(s, t)

In [13]:
laps.set_coordinate_system(cs)
laps.sectors.start_line = Segment(
    first=Point(x=-110, y=-420),
    second=Point(x=-104, y=-435),
)
laps.update()

In [14]:
fig = px.line(
    x=[cs.local(s).x for s in samples[500:]], y=[cs.local(s).y for s in samples[500:]]
)

laps.sectors.start_line = Segment(
    first=Point(x=-60, y=30),
    second=Point(x=-50, y=0),
)
fig.add_trace(
    px.line(
        x=[laps.sectors.start_line.first.x, laps.sectors.start_line.second.x],
        y=[laps.sectors.start_line.first.y, laps.sectors.start_line.second.y],
    ).data[0]
)

# # Play around to set right start line.
# s1 = Point(x=-110, y=-409)
# s2 = Point(x=-104, y=-415)

# fig.add_trace(px.line(x=[s1.x, s2.x], y=[s1.y, s2.y]).data[0])
fig.update_layout(height=600)

In [15]:
laps.update()

laps_times = pd.DataFrame(
    [dict(lap=i, lap_time=laps.lap_time(i)) for i in range(laps.laps_count())]
)

# Filter out laps around pits
non_outliars = laps_times["lap_time"] > 10

# Filter out slow laps: in-between sessions, bunch of yellows, etc.
non_outliars &= (
    laps_times["lap_time"] < 1.07 * laps_times.loc[non_outliars, "lap_time"].min()
)

px.line(
    laps_times.where(non_outliars),
    x="lap",
    y="lap_time",
    title=r"Lap times (whitin 107% of the best)",
    markers=True,
)

In [41]:
lap1 = laps.get_lap(39)
lap2 = laps.get_lap(33)

In [42]:
len(lap1.points), len(lap2.points)

(853, 854)

In [43]:
lap1.points[0].time, lap2.points[0].time

(1997.7483471817945, 1711.7003651993566)

In [44]:
locate_timestamp(lap1.points[0].time), locate_timestamp(lap1.points[-1].time)

(('/Users/denys/Documents/video-dump/GX020295.MP4',
  Timedelta('0 days 00:07:08.268347182')),
 ('/Users/denys/Documents/video-dump/GX020295.MP4',
  Timedelta('0 days 00:07:55.334688074')))

In [45]:
locate_timestamp(lap1.points[0].time), locate_timestamp(lap2.points[0].time)

(('/Users/denys/Documents/video-dump/GX020295.MP4',
  Timedelta('0 days 00:07:08.268347182')),
 ('/Users/denys/Documents/video-dump/GX020295.MP4',
  Timedelta('0 days 00:02:22.220365199')))

In [46]:
lap1.width = 10
lap2.width = 10
lap12 = lap2.resample(lap1, cs)
len(lap1.points), len(lap2.points), len(lap12.points)

(853, 854, 854)

In [47]:
def fix_xyz(point: GPSSample, xyz) -> GPSSample:
    gps = cs.global_(Vec3f(xyz["x"], xyz["y"], xyz["z"]))
    point.lat = gps.lat
    point.lon = gps.lon
    point.altitude = gps.altitude
    return point


new_points = [
    PointInTime_GPSSample(fix_xyz(point.point, corrected), point.time)
    for point, (_, corrected) in zip(
        lap1.points, (lap1_df + lap2_df.mean() - lap1_df.mean()).iterrows()
    )
]

NameError: name 'lap1_df' is not defined

In [48]:
from plotly.subplots import make_subplots


data = pd.DataFrame(
    dict(
        x=np.array(cs.local(p.point).x for p in lap12.points),
        y=np.array(cs.local(p.point).y for p in lap12.points),
        x2=np.array(cs.local(p.point).x for p in lap2.points),
        y2=np.array(cs.local(p.point).y for p in lap2.points),
        cum_dist=lap2.cum_distances,
        lap1_time=np.array([p.time - lap12.points[0].time for p in lap12.points]),
        lap2_time=np.array([p.time - lap2.points[0].time for p in lap2.points]),
        lap1_speed=np.array([p.point.full_speed for p in lap12.points]),
        lap2_speed=np.array([p.point.full_speed for p in lap2.points]),
    )
).assign(
    delta=lambda d: d["lap1_time"]
    - d["lap2_time"]
    - d["lap1_time"].min()
    + d["lap2_time"].min(),
    lap1_time=lambda d: d["lap1_time"],
    lap2_time=lambda d: d["lap2_time"],
)


fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        f"lap1_time ({lap1.lap_time():.3f}) - lap2_time ({lap2.lap_time():.3f}) (higher = lap1 is losing)",
        "Speed vs cum_dist",
    ),
)

# Delta trace (row 1)
fig.add_trace(
    px.line(data, x="cum_dist", y="delta", hover_data=["lap1_time", "lap2_time"]).data[
        0
    ],
    row=1,
    col=1,
)

# Speed traces (row 2)
speed_fig = px.line(data, x="cum_dist", y=["lap1_speed", "lap2_speed"])
for trace in speed_fig.data:
    fig.add_trace(trace, row=2, col=1)

fig.update_layout(
    height=600,
    margin=dict(l=0, r=0, t=30, b=0),
    legend=dict(orientation="h", y=-0.05),
)
display(fig)

In [49]:
lap1.lap_time(), lap2.lap_time()

(47.06634089224531, 46.969628067397935)

In [52]:
data

,x,y,x2,y2,cum_dist,lap1_time,lap2_time,lap1_speed,lap2_speed,delta
0,-55.219155,15.669716,-55.5681,16.713852,0.000000,0.000000,0.000000,18.610623,18.519602,0.000000
1,-55.047985,15.731859,-55.412068,16.769139,0.165565,0.009609,0.008822,18.635956,18.535000,0.000787
2,-54.058728,16.103467,-54.425383,17.113917,1.211761,0.065497,0.063822,18.722628,18.753000,0.001675
3,-53.061466,16.469353,-53.445647,17.480875,2.258364,0.121880,0.118822,18.745821,18.865000,0.003059
4,-52.078677,16.865298,-52.458962,17.858852,3.314971,0.176716,0.173822,18.926041,19.000000,0.002894
...,...,...,...,...,...,...,...,...,...,...
849,-58.77891,14.295332,-59.588095,16.269491,751.058463,46.863758,46.758798,18.455961,18.356000,0.104960
850,-57.816224,14.640121,-58.636152,16.669564,752.092035,46.917800,46.813798,18.606367,18.449000,0.104003
851,-56.877213,14.97663,-57.663364,17.047363,753.137159,46.971087,46.868798,18.650553,18.669000,0.102290
852,-55.918683,15.307156,-56.704473,17.403085,754.159936,47.025104,46.923798,18.665471,18.669000,0.101306


In [56]:
fig = px.scatter(
    data.assign(ddelta=lambda d: d["delta"].diff().rolling(20).mean().shift(-10)),
    x="x",
    y="y",
    color="ddelta",
    hover_data=["cum_dist", "lap1_speed", "lap2_speed"],
    title="(smoothed) Derivative of delta in space. Positive = lap1 is losing time",
).update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
for trace in (
    px.scatter(
        data.assign(ddelta=lambda d: -d["delta"].diff().rolling(20).mean().shift(-10)),
        x="x2",
        y="y2",
        color="ddelta",
        hover_data=["cum_dist", "lap1_speed", "lap2_speed"],
        title="(smoothed) Derivative of delta in space. Positive = lap1 is losing time",
    )
    .update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
    .data
):
    fig.add_trace(trace)
fig.update_layout(height=600)

In [55]:
fig = px.scatter(
    data.assign(ddelta=lambda d: d["lap1_speed"].diff().rolling(20).mean().shift(-10)),
    x="x",
    y="y",
    color="ddelta",
    hover_data=["cum_dist", "lap1_speed", "lap2_speed"],
    title="(smoothed) Acceleration (derivative of speed) of a lap",
).update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
for trace in (
    px.scatter(
        data.assign(ddelta=lambda d: d["lap2_speed"].diff().rolling(20).mean().shift(-10)),
        x="x2",
        y="y2",
        color="ddelta",
        hover_data=["cum_dist", "lap1_speed", "lap2_speed"],
        title="(smoothed) Derivative of delta in space. Positive = lap1 is losing time",
    )
    .update_layout(height=500, margin=dict(l=0, r=0, t=30, b=0))
    .data
):
    fig.add_trace(trace)
fig.update_layout(height=600)

In [ ]:
px.scatter(
    pd.DataFrame(
        dict(
            x=[cs.local(p.point).x for p in lap1.points],
            y=[cs.local(p.point).y for p in lap1.points],
            full_speed=[p.point.full_speed for p in lap1.points],
            cum_distance=lap1.cum_distances,
        )
    ),
    x="x",
    y="y",
    hover_data="cum_distance",
    color="delta",
).update_layout(height=600)

In [ ]:
lap1.lap_time(), lap21.lap_time()

In [ ]:
fig = px.line(x=[cs.local(s).x for s in samples], y=[cs.local(s).y for s in samples])
fig.add_trace(
    px.line(
        x=[laps.sectors.start_line.first.x, laps.sectors.start_line.second.x],
        y=[laps.sectors.start_line.first.y, laps.sectors.start_line.second.y],
    ).data[0]
)
fig.add_trace(px.line(x=[s1.x, s2.x], y=[s1.y, s2.y]).data[0])

In [ ]:
dir(laps.sectors)

In [ ]:
fig = px.line(x=[s.lat for s in samples], y=[s.lon for s in samples])

In [ ]:
df = pd.DataFrame(
    [
        dict(
            lat=s.lat,
            lon=s.lon,
            alt=s.altitude,
            full_speed=s.full_speed,
            ground_speed=s.ground_speed,
            timestamp=pd.to_datetime(s.timestamp_ms, unit="ms"),
            begin=b,
            end=e,
        )
        for s, b, e in samples
    ]
)

In [ ]:
df

In [ ]:
px.line((df["timestamp"] - df["timestamp"].min()).dt.total_seconds() - df["begin"])